# Fund static data enrichment & quality check

Load the **Funds** sheet : `CLEAN_RECOLTE DONNEES-EXCEL.xlsx`.

Funds in the import sheet are identified with an ISIN. As a first step we match the ISIN with a market identifier using the API of OpenFIGI, a service that maps proprietary or standard identifiers (like CUSIP, ISIN, or ticker symbols) to a FIGI identifier (an open, permanent, and unique 12-character alphanumeric standard used to identify financial instruments across global asset classes).

This allows then to retrieve data for the funds in EDGAR (Electronic Data Gathering, Analysis, and Retrieval system), the online public database and filing system operated by the U.S. Securities and Exchange Commission (SEC).

EDGAR data is used to enrich the dataset (with filing identifiers and prospectus fees) and to verify the quality of key attributes (`fund_wrapper`, `fund_type` and `benchmark_name`). The checks share one N-CEN census and one prospectus download per series.

Yahoo, Trackinsight and FMP (an online financial data platform and API provider that collects, cleans, and structures global market and fundamental data) data are also fetched to help completing and verifying the data.

In [ ]:
import sys
from pathlib import Path


def _ensure_helpers_importable() -> None:
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "helpers").is_dir():
            root = str(directory)
            if root not in sys.path:
                sys.path.insert(0, root)
            return
    raise FileNotFoundError(f"Could not find helpers/, starting from {start}")


_ensure_helpers_importable()

from helpers.funds_functions.load_workbook import (
    find_workbook,
    load_workbook,
    subgroup_counts,
)

workbook = find_workbook()
funds = load_workbook(workbook)
subgroup_counts_df = subgroup_counts(funds)
print("Counts of each (fund_wrapper, fund_type) subgroup:")
print(subgroup_counts_df)

### Match with OPENFIGI and retrieve market identifier

In [ ]:
from helpers.funds_functions.map_market_identifiers import map_market_identifiers

funds = map_market_identifiers(funds)

### Match OPENFIGI identifier with FinanceDatabase

In [ ]:
from helpers.funds_functions.map_financedatabase import map_financedatabase

funds = map_financedatabase(funds)

### Match OPENFIGI identifier with Yahoo

In [ ]:
from helpers.funds_functions.map_yahoo_static import map_yahoo_static

funds = map_yahoo_static(funds)


In [ ]:
from helpers.funds_functions.map_yahoo_timeseries import map_yahoo_timeseries

funds = map_yahoo_timeseries(funds)


### Match OPENFIGI identifier with EDGAR

In [ ]:
from helpers.funds_functions.map_edgar_tickers import (
    load_all_fund_reference_data,
    map_edgar_tickers,
)

# Load SEC fund reference (cache/cache-edgar-fund-data.json), download on first run.
fund_reference = load_all_fund_reference_data()
print(f"Fund reference columns: {', '.join(fund_reference.columns)}")

funds = map_edgar_tickers(funds, fund_reference=fund_reference)


### EDGAR N-CEN and prospectus facts

For each matched `edgartools_series_id`, fetch the **latest** N-CEN census and the latest matching prospectus (497K, then 485BPOS/485APOS). Only the fields needed for future wrapper, active/passive, benchmark, and TER checks are stored in `cache/cache-edgar-filing-facts.json`. This cell does not run any data-quality scores.


In [ ]:
import importlib
from helpers.funds_functions import fetch_edgar_filing_facts as filing_facts_mod

importlib.reload(filing_facts_mod)

funds = filing_facts_mod.fetch_edgar_filing_facts(funds)


### Fund wrapper and open-end proof (EDGAR)

For each fund, derive **open-end** status and **fund wrapper** (mutual fund vs ETF) from N-CEN in `cache-edgar-filing-facts.json`, with Yahoo then FMP as fallbacks when N-CEN cannot classify the wrapper. Lipper `fund_wrapper` is stored alongside each proof in `cache/dq-fund_wraper-edgar.json` for comparison.

In [ ]:
from helpers.funds_functions.dq_fund_wrapper_edgar import dq_fund_wrapper_edgar

funds = dq_fund_wrapper_edgar(funds)

In [ ]:
import importlib
from helpers.funds_functions import dq_fund_wrapper_edgar as dq_mod

importlib.reload(dq_mod)

wrapper_summary = dq_mod.summarize_lipper_edgar_wrapper_comparison()
print(f"Total funds in DQ cache: {wrapper_summary['total_funds']}")
print(f"Lipper matches EDGAR wrapper: {wrapper_summary['lipper_edgar_match']}")
print(f"Lipper differs from EDGAR wrapper: {wrapper_summary['lipper_edgar_mismatch']}")
print(
    "No EDGAR/Yahoo/FMP fund_wrapper classification: "
    f"{wrapper_summary['no_edgar_fund_wrapper']}"
)

mismatches = dq_mod.fund_wrapper_mismatches_dataframe()
if mismatches.empty:
    print("\nNo Lipper vs EDGAR wrapper mismatches.")
else:
    print(f"\n{mismatches.shape[0]} mismatch(es):")
    display(mismatches)

### Fund type proof (EDGAR)

For each fund, derive **Active** vs **Passive** from N-CEN `fundTypes` / `indexFundInfo` in `cache-edgar-filing-facts.json`. Lipper `fund_type` is stored alongside each proof in `cache/dq-fund_type-edgar.json` for comparison.

In [ ]:
import importlib
from helpers.funds_functions import dq_fund_type_edgar as dq_type_mod

importlib.reload(dq_type_mod)

funds = dq_type_mod.dq_fund_type_edgar(funds)

type_summary = dq_type_mod.summarize_lipper_edgar_fund_type_comparison()
print(f"Total funds in DQ cache: {type_summary['total_funds']}")
print(f"Lipper matches EDGAR fund_type: {type_summary['lipper_edgar_match']}")
print(f"Lipper differs from EDGAR fund_type: {type_summary['lipper_edgar_mismatch']}")
print(f"No N-CEN active/passive proof: {type_summary['no_edgar_fund_type']}")
print(
    "Lipper not labeled Active/Passive: "
    f"{type_summary['lipper_not_active_or_passive']}"
)

type_mismatches = dq_type_mod.fund_type_mismatches_dataframe()
if type_mismatches.empty:
    print("\nNo Lipper vs EDGAR fund_type mismatches.")
else:
    print(f"\n{type_mismatches.shape[0]} mismatch(es):")
    display(type_mismatches.head(20))

### S&P 500 benchmark proof (EDGAR)

For each fund, derive whether EDGAR shows a **core S&P 500** benchmark (total return, price, or unspecified) from prospectus `benchmark_indices`, with fallbacks to objective text and fund/series names in `cache-edgar-filing-facts.json`. Lipper `Benchmark Name` and `is_S&P500` are stored alongside each proof in `cache/dq-fund_benchmark-edgar.json` for comparison.

In [ ]:
import importlib
from helpers.funds_functions import dq_fund_benchmark_edgar as dq_bench_mod

importlib.reload(dq_bench_mod)

funds = dq_bench_mod.dq_fund_benchmark_edgar(funds)

bench_summary = dq_bench_mod.summarize_lipper_edgar_benchmark_comparison()
print(f"Total funds in DQ cache: {bench_summary['total_funds']}")
print(f"EDGAR S&P 500 benchmark proof: {bench_summary['edgar_sp500_benchmark_proof']}")
print(
    f"Lipper matches EDGAR S&P 500 presence: {bench_summary['lipper_edgar_presence_match']}"
)
print(
    f"Lipper differs from EDGAR S&P 500 presence: {bench_summary['lipper_edgar_presence_mismatch']}"
)
print(
    f"Lipper variant matches EDGAR: {bench_summary['lipper_edgar_variant_match']} "
    f"({bench_summary['lipper_edgar_variant_mismatch']} variant mismatches)"
)

bench_mismatches = dq_bench_mod.benchmark_mismatches_dataframe()
if bench_mismatches.empty:
    print("\nNo Lipper vs EDGAR S&P 500 presence mismatches.")
else:
    print(f"\n{bench_mismatches.shape[0]} presence mismatch(es):")
    display(bench_mismatches.head(20))

## Edgar-checks export

Builds a **new** workbook at `output-data/Lipper-static-data-mask.xlsx` from the in-memory `funds` frame and JSON caches only (no copy of the Lipper Excel file). It contains two sheets: **Edgar-checks** (data) and **Methodology** (sources and DQ rules, thesis-style text).

Sheet **Edgar-checks**: Lipper ISIN/name, EDGAR identification and filing fields (N-CEN + prospectus), and three DQ blocks (`fund_wrapper`, `fund_type`, S&P 500 benchmark) with Lipper vs EDGAR values, result, and justification.

Columns are grouped with a two-row header (section + field), color-coded section banners, and conditional highlighting (Match / Mismatch / Not comparable, missing ticker, zebra rows). Close the file in Excel before re-running this cell.


In [ ]:
import importlib
from helpers.funds_functions import edgar_checks_rows, write_edgar_checks_excel as edgar_export

importlib.reload(edgar_checks_rows)
importlib.reload(edgar_export)

output_path = edgar_export.export_edgar_checks(
    funds,
    workbook,
    destination="Lipper-static-data-mask.xlsx",
)
print(f"Saved {output_path}")